In [ ]:
import pandas as pd

In [ ]:
df_raw = pd.read_csv('./data/cirrhosis.csv').drop(columns=["ID", "N_Days"])

target_col = 'Status'

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from utils import create_evaluation_dataframe, Metric

In [ ]:
# Usunięcie rekordów z brakującymi wartościami kategorialnymi
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(exclude=["number"]).columns
df_raw = df_raw.dropna(subset=cat_cols)

X = df_raw.drop(target_col, axis=1)
y = df_raw[target_col]
y = LabelEncoder().fit_transform(y)

print(f"Kształt danych po usunięciu braków: {X.shape}")

In [ ]:
# Takie same wywołania jak w base_model_eval
X_train_cl_tmp, X_test, y_train_cl_tmp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_cl_tmp, y_train_cl_tmp, test_size=0.15/0.85, random_state=42)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")

In [ ]:
# Przygotowanie pipeline'ów
num_pipelines = {
    "mean_imputation": Pipeline([
        ("imputer", SimpleImputer(strategy="mean"))
    ])
}

cat_pipelines = {
    "one_hot": Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])
}

In [ ]:
# Przygotowanie modeli
models = {
    "GaussianNB_baseline": GaussianNB(),
    "GaussianNB_var_smoothing_1e-6": GaussianNB(var_smoothing=1e-6),
    "GaussianNB_var_smoothing_1e-12": GaussianNB(var_smoothing=1e-12)
}

In [ ]:
# Ewaluacja modeli
evaluation_df = create_evaluation_dataframe(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    num_pipelines=num_pipelines,
    cat_pipelines=cat_pipelines,
    models=models,
    sort_by=Metric.ACCURACY
)

evaluation_df

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipelines["mean_imputation"], X_train.select_dtypes(include=["number"]).columns),
        ("cat", cat_pipelines["one_hot"], X_train.select_dtypes(exclude=["number"]).columns),
    ]
)

best_model = GaussianNB()

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", best_model)
])

final_pipeline.fit(X_train, y_train)

y_train_pred = final_pipeline.predict(X_train)
y_val_pred = final_pipeline.predict(X_val)
y_test_pred = final_pipeline.predict(X_test)

def print_metrics(y_true, y_pred, set_name):
    print(f"--- {set_name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}\n")

print_metrics(y_train, y_train_pred, "Train")
print_metrics(y_val, y_val_pred, "Validation")
print_metrics(y_test, y_test_pred, "Test")